# Classificando nomes com uma *Character-Level RNN*

Esse notebook foi criado com base no tutorial do PyTorch: <br> 
https://github.com/spro/practical-pytorch/blob/master/char-rnn-classification/char-rnn-classification.ipynb

### Problema: Dado um nome prÃ³prio de entrada, classificar esse nome de acordo com a nacionalidade a que ele pertence.

Entrada: **Hinton**

(-0.47) Scottish

(-1.52) English

(-3.57) Irish


-

Entrada: **Schmidhuber**

(-0.19) German

(-2.48) Czech

(-2.68) Dutch

### Import de bibliotecas

In [ ]:
# Para os dados
import unicodedata
import string
import sys, random, os

# Para o processamento e anÃ¡lise
import torch
from torch import nn
import numpy as np
import seaborn as sns
sns.set_style('darkgrid')
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
args = {
    'lr': 5e-5,
    'regularizacao': 1e-7,
    'num_epocas': 40,
}
args['device'] = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## Dados de entrada

### Importando dataset

In [ ]:
# comente as duas linhas seguintes caso rode mais de uma vez
!wget https://download.pytorch.org/tutorial/data.zip #
!unzip data.zip #
############################################################

In [ ]:
# Transforma um arquivo de nomes em listas e/ou arrays (nomes, rótulos)
def readLines(filename):
    with open(filename, encoding='utf-8') as arquivo:
        linhas = arquivo.read().strip().splitlines()

    # Remove acentos e mantém os nomes como texto (str)
    nomes = [
        unicodedata.normalize('NFKD', linha)
        .encode('ascii', 'ignore')
        .decode('ascii')
        for linha in linhas
    ]

    # Funciona corretamente tanto no Windows quanto em outros sistemas
    categoria = os.path.splitext(os.path.basename(filename))[0]
    rotulos = np.repeat(categoria, len(nomes))

    return nomes, rotulos


root_path = 'data/names'
arquivos = sorted(os.listdir(root_path))
categorias = [os.path.splitext(arquivo)[0] for arquivo in arquivos]

dados, rotulos = [], []
samples_perclass = []

for file_name in arquivos:
    caminho = os.path.join(root_path, file_name)
    nomes, etiquetas = readLines(caminho)

    dados.append(nomes)
    rotulos.append(etiquetas)
    samples_perclass.append((file_name, len(nomes)))


print(samples_perclass)
print('Mínimo de amostras:', min(samples_perclass, key=lambda item: item[1]), '\n')

indice_portugues = categorias.index('Portuguese')
print(dados[indice_portugues][:10])
print(rotulos[indice_portugues][:10])


### Convertendo os dados para tensor

**Convertendo os rÃ³tulos para tensor**

RepresentaÃ§Ã£o One-Hot de 18 categorias de idiomas que queremos prever.

In [ ]:
def label2tensor(rotulos):
  rotulos_tns = torch.zeros( len(rotulos), 1, dtype=torch.int64 )
  for k, rotulo in enumerate(rotulos):
    idx = categorias.index(rotulo)
    rotulos_tns[k][0] = idx
  return rotulos_tns

rotulos_arabe = rotulos[0]
rotulos_tns = label2tensor(rotulos_arabe)
print(type(rotulos_tns), rotulos_arabe[0], rotulos_tns[0])

**Convertendo os nomes para tensor**

Aqui tambÃ©m usaremos a representaÃ§Ã£o One-Hot, porÃ©m teremos que trabalhar com uma lista de tensores, pois os nomes tem comprimentos diferentes. Mais Ã  frente no curso aprenderemos a lidar com isso da forma certa!

In [ ]:
caracteres_validos = string.ascii_letters
print(caracteres_validos)
tam_dicionario = len(caracteres_validos)

def nome2tensor(nome):
  tns = torch.zeros( len(nome), tam_dicionario )
  
  for k, letra in enumerate(nome):
    idx = caracteres_validos.find(letra)
    tns[k, idx] = 1

  return tns

dados_arabe = dados[0]
dados_tns = [nome2tensor(dado) for dado in dados_arabe]
print(dados_arabe[0][0],'\n', dados_tns[0][0])


**Amostrando batch balanceado**

In [ ]:
num_amostras = min(samples_perclass, key= lambda k: k[1])[1]

def sample_batch(size=num_amostras):
  dados_batch, rotulos_batch = [], []
  for cat in categorias:
    
    amostras_cat = dados[categorias.index(cat)]
    idx = np.random.choice(range(len(amostras_cat)), size=size)
    
    dados_batch.extend([ r for k, r in enumerate(dados[categorias.index(cat)]) if k in idx])
    rotulos_batch.extend([ r for k, r in enumerate(rotulos[categorias.index(cat)]) if k in idx])

  dados_tns = [nome2tensor(dado) for dado in dados_batch]
  return dados_tns, label2tensor(rotulos_batch)

dados_batch, rotulos_batch = sample_batch()
print(len(dados_batch), dados_batch[0].size(), rotulos_batch.size())

### Por fim, vamos separar dados de treino e teste.

In [ ]:

Xtrain, Xtest, Ytrain, Ytest = split(dados_tns, rotulos_tns, test_size=0.2)
print(len(Xtrain), len(Xtest))

## Modelo Recorrente

* Implemente um modelo para classificaÃ§Ã£o de nomes prÃ³prios (sÃ©rie de caracteres) usando apenas camadas *RNNCell*, *Linear* e ativaÃ§Ã£o *LogSoftmax*
* Cada entrada (caracter) possui dimensÃ£o (52): alfabeto maiÃºsculo e minÃºsculo
* *Hidden size* possui dimensÃ£o (256): hiperparÃ¢metro 
* SaÃ­da possui dimensÃ£o (18): vetor de probabilidade de classes
* Batch size = 1 **pra nÃ£o termos que lidar com as sequÃªncias de tamanho variÃ¡vel.**

### Links Ãºteis

RNNCell: https://pytorch.org/docs/stable/generated/torch.nn.RNNCell.html#torch.nn.RNNCell

Linear: https://pytorch.org/docs/stable/generated/torch.nn.Linear.html#torch.nn.Linear

Non-linear activations: https://pytorch.org/docs/stable/generated/torch.nn.LogSoftmax.html#torch.nn.LogSoftmax

In [ ]:
class RNN(nn.Module):
    def __init__(self, tam_entrada, tam_feature, tam_saida):
        super(RNN, self).__init__()
        
        self.tam_entrada = tam_entrada
        self.tam_feature = tam_feature
        self.tam_saida   = tam_saida
        
        self.rnn    = nn.RNNCell(self.tam_entrada, self.tam_feature)
        self.linear = nn.Linear(self.tam_feature, self.tam_saida)
        self.softmax = nn.LogSoftmax(dim=1)
    
    def forward(self, nome):
      
        # Inicialize o estado interno da RNN
        hidden = torch.zeros(1, self.tam_feature, device=nome.device)
        
        for letra in nome:
          letra = letra.unsqueeze(0)
          hidden = self.rnn(letra, hidden)
        
        saida = self.linear(hidden)
        saida = self.softmax(saida) 
        return saida

tam_feature = 256
model = RNN(tam_dicionario, tam_feature, len(categorias))
model.to(args['device'])

## Loss e Otimizador

In [ ]:
criterion = nn.NLLLoss().to(args['device']) 
optimizer = torch.optim.Adam(model.parameters(), lr=args['lr'], weight_decay=args['regularizacao'])

## Treinamento

A otimizaÃ§Ã£o Ã© um processo que tem uma raÃ­z muito bem definida de passo a passo, sempre fazemos:
* Carregar os dados e colocar no dispositivo de hardware adequado
* Forward do dado na rede
* CÃ¡lculo da funÃ§Ã£o de custo (no nosso caso uma funÃ§Ã£o composta)
* Passos de OtimizaÃ§Ã£o
  * Zerar os gradientes do otimizador (`optimizer.zero_grad()`)
  * Calcular os gradientes com base na loss (`loss.backward()`)
  * Passo de otimizaÃ§Ã£o (`optimizer.step()`)

Apesar de cada soluÃ§Ã£o ter pequenas variaÃ§Ãµes em um ou mais passos do fluxo, o esqueleto Ã© sempre o mesmo. 

In [ ]:
def forward(X, Y, etapa):
  if etapa == 'Treino': model.train()
  else: model.eval()

  acuracia = 0.
  loss_epoca = []
  for k, (dado, rotulo) in enumerate(zip(X, Y)):
      
    dado = dado.to(args['device'])
    rotulo = rotulo.to(args['device'])

    saida = model(dado)
    loss = criterion(saida, rotulo)
    loss_epoca.append(loss.detach().cpu().numpy())

    _, pred = torch.max(saida, axis=-1)
    acuracia += 1 if pred[0].item() == rotulo[0].item() else 0

    if etapa == 'Treino':
      # OtimizaÃ§Ã£o
      optimizer.zero_grad()
      loss.backward()
      optimizer.step()

  loss_epoca = np.asarray(loss_epoca).ravel()
  acuracia   = acuracia/float(len(loss_epoca))
  print('\n','*'*15 + etapa + '*'*15 )
  print('Epoca: {:}, Loss: {:.4f} +/- {:.4f}, AcurÃ¡cia: {:.4f}'.format(epoca, loss_epoca.mean(), 
                                                                        loss_epoca.std(), 
                                                                        acuracia
                                                                       )) 
  return loss_epoca.mean(), acuracia

In [ ]:
loss_treino, loss_test = [], []
acc_treino, acc_test = [], []

dados_test, rotulos_test = sample_batch(size=5)
for epoca in range(200):

  dados_tns, rotulos_tns = sample_batch()

  loss, acuracia = forward(dados_tns, rotulos_tns, 'Treino')
  loss_treino.append(loss)
  acc_treino.append(acuracia)  

  loss, acuracia = forward(dados_test, rotulos_test, 'Teste')
  loss_test.append(loss)
  acc_test.append(acuracia)


## AnÃ¡lise de ConvergÃªncia

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12,3))

ax1.plot(loss_treino[1:], label='Train')
ax1.plot(loss_test[1:], label='Test')
ax1.set_title('Model Convergence - Loss')
ax1.set_xlabel('epochs')
ax1.set_ylabel('Loss')
ax1.legend()

ax2.plot(acc_treino, label='Train')
ax2.plot(acc_test, label='Test')
ax2.set_title('Model Convergence - Accuracy')
ax2.set_xlabel('epochs')
ax2.set_ylabel('Accuracy')
ax2.legend()

## Usando o modelo

In [ ]:
def predict(nome):
  model.eval()

  tns = torch.zeros( len(nome), tam_dicionario )
  for k, letra in enumerate(nome):
    idx = caracteres_validos.find(letra)
    tns[k, idx] = 1
  tns = tns.to(args['device'])

  saida = model(tns)
  topv, topi = saida.data.topk(3, 1, True)

  print(nome)
  for value, index in zip(topv[0], topi[0]):
    print('(%.2f) %s' % (value, categorias[index]))
  print('\n')
  
predict('Mota')
predict('Kamikaze')
predict('Suarez')

In [ ]:
import torch

entrada_teste = torch.zeros(
    5, tam_dicionario,
    device=args['device']
)

resultado = model(entrada_teste)

print("Formato da saída:", resultado.shape)
print("Quantidade de nacionalidades:", len(categorias))